# SnapClass — Research & Trials Notebook

Use this notebook to experiment with pipelines, thresholds, and database calls **before** touching the modular code.

**Pre-requisite:** run `pip install -r requirements.txt` from the project root once — the `-e .` line makes all `src.*` imports work here automatically.

---
## Table of Contents
1. [Environment Check](#1)
2. [Face Pipeline — Embedding Extraction](#2)
3. [Face Pipeline — Threshold Tuning](#3)
4. [Voice Pipeline — Embedding Extraction](#4)
5. [Voice Pipeline — Speaker Similarity](#5)

---
<a id='1'></a>
## 1. Environment Check

In [1]:
import sys, torch, numpy as np
print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("NumPy  :", np.__version__)
print("CUDA   :", torch.cuda.is_available())

Python : 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.2.2+cpu
NumPy  : 1.26.4
CUDA   : False


---
<a id='2'></a>
## 2. Face Pipeline — Embedding Extraction

Test `get_face_embeddings()` on a local image without touching Streamlit.

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# ── Load a test image (swap with your own path) ──────────────────────────────
IMG_PATH = "../assets/test_face.jpg"   # <-- change this

img = Image.open(IMG_PATH).convert("RGB")
img_np = np.array(img)

plt.figure(figsize=(4, 4))
plt.imshow(img_np)
plt.axis("off")
plt.title("Input image")
plt.show()

print("Image shape:", img_np.shape)

In [ ]:
# NOTE: get_face_embeddings uses @st.cache_resource internally.
# Outside Streamlit we call the underlying models directly.
from facenet_pytorch import MTCNN, InceptionResnetV1

mtcnn  = MTCNN(keep_all=True, device='cpu', post_process=True)
resnet = InceptionResnetV1(pretrained='vggface2').eval()

faces = mtcnn(img)   # returns list of (3,160,160) tensors or None

if faces is None:
    print("❌ No face detected")
else:
    print(f"✅ Detected {len(faces)} face(s)")
    embeddings = [resnet(f.unsqueeze(0)).detach().numpy()[0] for f in faces]
    print(f"   Embedding shape: {embeddings[0].shape}")
    print(f"   Embedding norm : {np.linalg.norm(embeddings[0]):.4f}")

---
<a id='3'></a>
## 3. Face Pipeline — Threshold Tuning

Compare L2 distances between **same-person** and **different-person** embeddings to pick the best threshold (default is `0.9`).

In [ ]:
import matplotlib.pyplot as plt

# ── Swap these with real test image paths ─────────────────────────────────────
SAME_PERSON_IMAGES    = ["../assets/alice_1.jpg", "../assets/alice_2.jpg"]
DIFF_PERSON_IMAGES    = ["../assets/alice_1.jpg", "../assets/bob_1.jpg"]

def embed(path):
    img = np.array(Image.open(path).convert("RGB"))
    faces = mtcnn(Image.fromarray(img))
    if faces is None:
        raise ValueError(f"No face in {path}")
    return resnet(faces[0].unsqueeze(0)).detach().numpy()[0]

try:
    e1_same, e2_same = embed(SAME_PERSON_IMAGES[0]), embed(SAME_PERSON_IMAGES[1])
    e1_diff, e2_diff = embed(DIFF_PERSON_IMAGES[0]), embed(DIFF_PERSON_IMAGES[1])

    dist_same = np.linalg.norm(e1_same - e2_same)
    dist_diff = np.linalg.norm(e1_diff - e2_diff)

    print(f"Same person L2 distance : {dist_same:.4f}  (should be < 0.9)")
    print(f"Diff person L2 distance : {dist_diff:.4f}  (should be > 0.9)")

    plt.barh(["same person", "diff person"], [dist_same, dist_diff], color=["green", "red"])
    plt.axvline(x=0.9, color="black", linestyle="--", label="threshold=0.9")
    plt.xlabel("L2 distance")
    plt.legend()
    plt.title("Face distance — threshold check")
    plt.show()
except Exception as e:
    print(f"Skipped (images not found): {e}")

---
<a id='4'></a>
## 4. Voice Pipeline — Embedding Extraction

In [ ]:
import io, torch, librosa, numpy as np
from speechbrain.pretrained import EncoderClassifier

encoder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": "cpu"}
)
print("✅ ECAPA-TDNN loaded")

In [ ]:
# ── Swap with a real .wav / .mp3 path ────────────────────────────────────────
AUDIO_PATH = "../assets/test_voice.wav"   # <-- change this

try:
    with open(AUDIO_PATH, "rb") as f:
        audio_bytes = f.read()

    audio, sr = librosa.load(io.BytesIO(audio_bytes), sr=16000, mono=True)
    wav        = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)
    emb        = encoder.encode_batch(wav).squeeze().detach().numpy()
    emb        = emb / (np.linalg.norm(emb) + 1e-8)   # L2-normalise

    print(f"Embedding shape: {emb.shape}")
    print(f"Embedding norm : {np.linalg.norm(emb):.4f}  (should be ~1.0 after normalisation)")
except FileNotFoundError as e:
    print(f"Skipped — file not found: {e}")

---
<a id='5'></a>
## 5. Voice Pipeline — Speaker Similarity

Verify cosine similarity between same/different speakers and tune the `0.75` threshold.

In [ ]:
from src.pipelines.voice_pipeline import identify_speaker

# ── Replace with real audio bytes from two speakers ──────────────────────────
# embedding_alice_1 = ...   (numpy array, shape (192,), L2-normed)
# embedding_alice_2 = ...   (same person, different recording)
# embedding_bob     = ...   (different person)

# Simulate with random vectors to verify the API shape
np.random.seed(42)
base     = np.random.randn(192);  base /= np.linalg.norm(base)
same     = base + np.random.randn(192) * 0.05;  same /= np.linalg.norm(same)   # very similar
diff     = np.random.randn(192);  diff /= np.linalg.norm(diff)                   # random

candidates = {"alice": base.tolist(), "bob": diff.tolist()}

sid, score = identify_speaker(same.tolist(), candidates, threshold=0.75)
print(f"Matched: {sid}  |  cosine similarity: {score:.4f}")

sid2, score2 = identify_speaker(diff.tolist(), candidates, threshold=0.75)
print(f"Matched: {sid2}  |  cosine similarity: {score2:.4f}")